# Notebook for Distribution and count all the free text annotation

#### Import Packages

In [4]:
import os
import json
import pandas as pd
from collections import defaultdict

#### Pipeline for extract free text annotation type distribution

In [5]:
def extract_annotation_distribution(
    raw_base="Raw_free_text_annotation",
    output_base="Distribution_free_text_annotation"
):
    """
    Build structured annotation distribution:
    io_type | annotation | tools | tool_count
    """

    domains = [
        "Genetic_variant",
        "Metagenomic",
        "Microscopy",
        "Neuroimaging",
        "Phylogeny",
        "Single_Cell",
        "Systems_biology",
    ]

    field_mapping = {
        "input data": "Input",
        "output data": "Output",
        "input format": "Input_format",
        "output format": "Output_format",
        "topic": "Topic",
        "operation": "Operation",
    }

    for domain in domains:
        print(f"\nProcessing domain: {domain}")

        json_file = os.path.join(
            raw_base,
            domain,
            f"{domain}_tools_annotations.json"
        )

        if not os.path.exists(json_file):
            print(f"Missing file: {json_file}")
            continue

        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Output folder
        output_domain_folder = os.path.join(output_base, domain)
        os.makedirs(output_domain_folder, exist_ok=True)

        # COLLECT STRUCTURED DATA
        # key = (io_type, annotation) to set(tools)
        distribution = defaultdict(set)

        for tool_key, tool_data in data.items():

            if tool_data.get("source") != "Consensus expert LLM":
                continue

            tool_name = tool_data.get("name", "").strip()

            for json_field, io_type in field_mapping.items():
                annotations = tool_data.get(json_field, [])

                for ann in annotations:
                    ann_clean = ann.strip().lower()
                    if ann_clean:
                        distribution[(io_type, ann_clean)].add(tool_name)

        # BUILD DATAFRAME
        rows = []

        for (io_type, annotation), tools in distribution.items():
            tool_list = sorted(tools)

            rows.append({
                "io_type": io_type,
                "annotation": annotation,
                "tools": ", ".join(tool_list),
                "tool_count": len(tool_list),
            })

        df = pd.DataFrame(rows)

        df = df.sort_values(by=["io_type", "annotation"])

        # SAVE CSV
        output_file = os.path.join(
            output_domain_folder,
            f"{domain}_annotation_distribution.csv"
        )

        df.to_csv(output_file, index=False)

        print(f" Saved: {output_file}")


In [6]:
extract_annotation_distribution()


Processing domain: Genetic_variant
 Saved: Distribution_free_text_annotation/Genetic_variant/Genetic_variant_annotation_distribution.csv

Processing domain: Metagenomic
 Saved: Distribution_free_text_annotation/Metagenomic/Metagenomic_annotation_distribution.csv

Processing domain: Microscopy
 Saved: Distribution_free_text_annotation/Microscopy/Microscopy_annotation_distribution.csv

Processing domain: Neuroimaging
 Saved: Distribution_free_text_annotation/Neuroimaging/Neuroimaging_annotation_distribution.csv

Processing domain: Phylogeny
 Saved: Distribution_free_text_annotation/Phylogeny/Phylogeny_annotation_distribution.csv

Processing domain: Single_Cell
 Saved: Distribution_free_text_annotation/Single_Cell/Single_Cell_annotation_distribution.csv

Processing domain: Systems_biology
 Saved: Distribution_free_text_annotation/Systems_biology/Systems_biology_annotation_distribution.csv


#### Count annotation type

In [13]:
domains = [
    "Metagenomic",
    "Neuroimaging",
    "Phylogeny",
    "Single_Cell",
    "Systems_biology",
    "Genetic_variant",
    "Microscopy"
]

base_folder = "Distribution_free_text_annotation"

columns = ["Data", "Format", "Topic", "Operation"]

df_counts = pd.DataFrame(index=domains, columns=columns)

# Global sets per type
global_sets = {
    "Data": set(),
    "Format": set(),
    "Topic": set(),
    "Operation": set(),
}

for domain in domains:

    file_path = os.path.join(
        base_folder,
        domain,
        f"{domain}_annotation_distribution.csv"
    )

    if not os.path.exists(file_path):
        print(f"Missing: {file_path}")
        df_counts.loc[domain] = 0
        continue

    df = pd.read_csv(file_path)

    df["annotation"] = df["annotation"].str.strip().str.lower()
    df["io_type"] = df["io_type"].str.strip()

    # DOMAIN LEVEL COUNTS
    data_set = set(
        df[df["io_type"].isin(["Input", "Output"])]["annotation"]
    )

    format_set = set(
        df[df["io_type"].isin(["Input_format", "Output_format"])]["annotation"]
    )

    topic_set = set(
        df[df["io_type"] == "Topic"]["annotation"]
    )

    operation_set = set(
        df[df["io_type"] == "Operation"]["annotation"]
    )

    df_counts.loc[domain] = [
        len(data_set),
        len(format_set),
        len(topic_set),
        len(operation_set),
    ]

    global_sets["Data"].update(data_set)
    global_sets["Format"].update(format_set)
    global_sets["Topic"].update(topic_set)
    global_sets["Operation"].update(operation_set)

df_counts = df_counts.astype(int)

df_counts.loc["Total"] = df_counts.sum(axis=0)

# UNIQUE ACROSS DOMAINS
df_counts.loc["Unique_all_domains"] = [
    len(global_sets["Data"]),
    len(global_sets["Format"]),
    len(global_sets["Topic"]),
    len(global_sets["Operation"]),
]

# Number of tools
df_counts["Number of tools"] = [8, 5, 3, 3, 11, 11, 5, 45, 45]

# OUTPUT
df_counts



,Data,Format,Topic,Operation,Number of tools
Metagenomic,24,10,15,25,8
Neuroimaging,17,25,10,10,5
Phylogeny,9,5,8,8,3
Single_Cell,20,11,4,14,3
Systems_biology,76,44,49,65,11
Genetic_variant,28,19,16,19,11
Microscopy,17,14,9,14,5
Total,191,128,111,155,45
Unique_all_domains,184,97,105,146,45


In [14]:
df_counts.to_csv("Distribution_free_text_annotation/annotation_counts_summary_final.csv")